In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
image_dir = f"{base_dir}/AntM2C_image_structured"
preprocessed_dir = f"{base_dir}/AntM2C_preprocessed"
image_metadata_path = f"{preprocessed_dir}/image_paths.csv"

In [3]:
# Import libraries
import os
import pandas as pd
import numpy as np
import torch

from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
from torch.utils.data import DataLoader
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModel

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
# Save a copy of the original image_paths.csv
image_metadata_path = f"{base_dir}/AntM2C_preprocessed/image_paths.csv"
backup_path = image_metadata_path.replace(".csv", "_backup.csv")

!cp "{image_metadata_path}" "{backup_path}"
print(f"Backup saved to {backup_path}")

Backup saved to /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/image_paths_backup.csv


In [ ]:
# Create a new destination folder for organized images
new_image_dir = os.path.join(base_dir, "AntM2C_image_structured")
os.makedirs(new_image_dir, exist_ok=True)

In [ ]:
import shutil
df_images = pd.read_csv(image_metadata_path)
moved_count = 0
new_paths = []

In [ ]:
for _, row in tqdm(df_images.iterrows(), total=len(df_images), desc="Copying to structured image dir"):
    item_id = str(row['item_id'])
    file_name = f"{item_id}.png"

    subfolder = item_id[:3].zfill(3)
    subfolder_path = os.path.join(new_image_dir, subfolder)
    try:
        os.makedirs(subfolder_path, exist_ok=True)
    except Exception as e:
        print(f"Error creating folder {subfolder_path}: {e}")
        new_paths.append(None)
        continue

    src = os.path.join(image_dir, file_name)
    dst = os.path.join(subfolder_path, file_name)

    if os.path.exists(src):
        try:
            shutil.copy2(src, dst)
            new_paths.append(f"{subfolder}/{file_name}")
            moved_count += 1
        except Exception as e:
            print(f"Error copying {file_name}: {e}")
            new_paths.append(None)
    else:
        new_paths.append(None)

print(f"Successfully copied {moved_count:,} images to structured subfolders.")

Copying to structured image dir:  91%|█████████ | 53636/59142 [4:35:26<3:00:15,  1.96s/it]

In [ ]:
!ls -lh "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"

In [ ]:
!ls -lh "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_image_structured"

In [16]:
# Define base directory
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
structured_dir = Path(f"{base_dir}/AntM2C_image_structured")

In [17]:
# Traverse subfolders and collect item_id + image_path
image_entries = []
for path in structured_dir.rglob("*.png"):
    item_id = path.stem
    rel_path = path.relative_to(structured_dir).as_posix()
    image_entries.append({"item_id": item_id, "image_path": rel_path})

In [18]:
# Create DataFrame and save
df_structured_paths = pd.DataFrame(image_entries)
output_csv = f"{base_dir}/AntM2C_preprocessed/image_paths_structured.csv"
df_structured_paths.to_csv(output_csv, index=False)

In [21]:
print(f"Saved updated metadata with {len(df_structured_paths):,} entries -> {output_csv}")
df_structured_paths.head()

Saved updated metadata with 59,142 entries -> /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/image_paths_structured.csv


,item_id,image_path
0,28721,287/28721.png
1,287697,287/287697.png
2,287443,287/287443.png
3,28734,287/28734.png
4,287655,287/287655.png


At this point, setup is done, and we will be moving on to;
## Setting up for Image Preprocessing and Batching
(efficient batching + corrupted-image skip logic)

In [6]:
def load_and_preprocess(path):
    """
    Loads an image from disk and applies ViT preprocessing.
    Safely handles unreadable or corrupted images.
    """
    try:
        img = Image.open(path).convert("RGB")
    except (FileNotFoundError, UnidentifiedImageError, OSError):
        return None  # treat as corrupted/unreadable

    try:
        pixel_values = processor(img, return_tensors="pt")["pixel_values"]
        return pixel_values
    except Exception:
        return None

In [7]:
def batched(iterable, batch_size):
    """
    Simple batching generator.
    """
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

In [22]:
%cd "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"

/content/drive/.shortcut-targets-by-id/1J-_B9dYFbtgqYmeg_SCsCplymxFmH7Pn/Final_Year_Project/Implementation/datasets/AntM2C_10M


In [23]:
df = pd.read_csv("AntM2C_preprocessed/image_paths_structured.csv")
print("Total entries:", len(df))

valid_items = []
missing_count = 0

Total entries: 59142


In [20]:
import os

print("image_dir =", image_dir)
print("Base dir exists?    ", os.path.exists(image_dir))

print("\nFirst few rows from df:")
print(df.head())

first = df.iloc[0]
print("\nRaw image_path in first row (repr):", repr(first["image_path"]))

test_full_path = os.path.join(image_dir, str(first["image_path"]))
print("Test full path:", test_full_path)
print("Exists?", os.path.exists(test_full_path))

image_dir = /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_image_structured
Base dir exists?     True

First few rows from df:
   item_id      image_path
0    28721   287/28721.png
1   287697  287/287697.png
2   287443  287/287443.png
3    28734   287/28734.png
4   287655  287/287655.png

Raw image_path in first row (repr): '287/28721.png'
Test full path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_image_structured/287/28721.png
Exists? True


In [19]:
# Ensure clean, normalised paths with remove spaces/newlines and remove leading / or \ if present
df["image_path"] = (
    df["image_path"]
    .astype(str)
    .str.strip()
    .str.lstrip("/\\")
)

# make sure item_id is int
df["item_id"] = df["item_id"].astype(int)

In [12]:
valid_items = []
missing_count = 0
debug_shown = 0  # only show a few missing for inspection

for _, row in tqdm(df.iterrows(), total=len(df)):
    item_id = int(row["item_id"])
    img_path = row["image_path"]          # already cleaned above
    full_path = os.path.join(image_dir, img_path)

    if not os.path.exists(full_path):
        missing_count += 1
        # show first few missing paths so you can visually check
        if debug_shown < 5:
            print("[MISSING SAMPLE]", repr(full_path))
            debug_shown += 1
        continue

    valid_items.append((item_id, full_path))

print(f"Valid images found: {len(valid_items)}")
print(f"Missing images skipped: {missing_count}")

100%|██████████| 59142/59142 [01:57<00:00, 502.76it/s] 

Valid images found: 59142
Missing images skipped: 0
